In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path('..').resolve()
FISHER_TAB = ROOT / 'results' / 'fisher_day0_day15' / 'tables'
VDJDB = ROOT / 'results' / 'fisher_day0_day15_extended' / 'databases' / 'vdjdb-2026-06-03' / 'vdjdb.slim.txt'
YFV_CSV = ROOT / 'results' / 'day0_day15_external_tcr_annotation' / 'databases' / 'Yellow_Fever_TCRs.csv'
OUT = ROOT / 'coursework_submission_notebooks' / 'outputs' / '07_annotation'
TAB = OUT / 'tables'
FIG = OUT / 'figures'
TAB.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)
DONORS = ['P1', 'P2', 'Q1', 'Q2', 'S1', 'S2']
YFV_PAT = re.compile('yellow\\s*fever|\\byfv\\b|17d', re.I)


In [ ]:
yfv_aa = set()
if VDJDB.exists():
    vdj = pd.read_csv(VDJDB, sep='\t', usecols=['gene', 'cdr3', 'antigen.species', 'antigen.epitope'], low_memory=False)
    mask = vdj['gene'].astype(str).str.upper().eq('TRB') & vdj['antigen.species'].astype(str).str.contains(YFV_PAT, na=False)
    yfv_aa |= set(vdj.loc[mask, 'cdr3'].dropna().astype(str))
    epitopes = vdj.loc[mask].groupby('cdr3')['antigen.epitope'].apply(lambda x: ';'.join(sorted(set(map(str, x)))[:5])).to_dict()
else:
    epitopes = {}
if YFV_CSV.exists():
    yf = pd.read_csv(YFV_CSV)
    col = next((c for c in yf.columns if c.lower() in ('cdr3', 'cdr3_aa', 'cdr3aa')), None)
    if col:
        yfv_aa |= set(yf[col].dropna().astype(str))
yfv_aa = {a for a in yfv_aa if isinstance(a, str) and len(a) >= 5 and a.startswith('C')}


In [ ]:
rows = []
match_rows = []
for donor in DONORS:
    f = pd.read_csv(FISHER_TAB / f'fisher_positive_FC4_{donor}.csv')
    hit = f['cdr3_aa'].astype(str).isin(yfv_aa)
    n = len(f)
    n_hit = int(hit.sum())
    rows.append({'donor': donor, 'n_fisher': n, 'n_exact_YFV': n_hit, 'pct_exact_YFV': 100.0 * n_hit / n if n else np.nan})
    sub = f[hit].copy()
    sub['epitope'] = sub['cdr3_aa'].map(lambda x: epitopes.get(x, ''))
    match_rows.append(sub)
summary = pd.DataFrame(rows)
matches = pd.concat(match_rows, ignore_index=True) if match_rows else pd.DataFrame()
summary.to_csv(TAB / 'exact_YFV_summary_by_donor.csv', index=False)
matches.to_csv(TAB / 'exact_YFV_Fisher_markers.csv', index=False)
summary


In [ ]:
(fig, ax) = plt.subplots(figsize=(8, 4))
x = np.arange(len(summary))
ax.bar(x, summary['n_exact_YFV'], color='#2ca02c')
ax.set_xticks(x)
ax.set_xticklabels(summary['donor'])

for (i, r) in summary.iterrows():
    ax.text(i, r['n_exact_YFV'] + 1, f"{r['pct_exact_YFV']:.1f}%", ha='center', fontsize=8)
plt.tight_layout()
plt.savefig(FIG / 'exact_YFV_counts.png', dpi=150, bbox_inches='tight')
plt.show()
